## PACOTES 

In [5]:
import time
import itertools
import numpy as np
import pandas as pd

from joblib import Parallel, delayed

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve,
    auc,
    matthews_corrcoef,
    log_loss
)

from scipy.stats import ks_2samp

## CONFIGURACOES

In [6]:

# CONFIG
inicio = time.time()

TARGET_COL = "status_fraude"
THRESHOLD = 0.50
estado_randomico = 42

numero_de_componentes = 2
inicializacoes_gausianas = 3
tipo_matriz_covariancia = "full"
erro_numerico = 1e-6

N_JOBS = 7

NOME_CSV = "3x3_visu_scores.csv"

## CRIACAO DO CSV

In [7]:
# LOAD DATASET
df = pd.read_csv("creditcard.csv")

features = [
    col for col in df.columns
    if col != TARGET_COL
]

combinacoes = list(itertools.combinations(features, 3))

print(f"Total de trios: {len(combinacoes)}")

# FUNÇÃO SCORE FINAL
def calcular_score_final(auc_pr, mcc, ks, ll):

    auc_pr_norm = np.clip(auc_pr, 0, 1)

    mcc_norm = (mcc + 1) / 2
    mcc_norm = np.clip(mcc_norm, 0, 1)

    ks_norm = np.clip(ks, 0, 1)

    log_loss_norm = 1 / (1 + ll)

    score_final = (
        auc_pr_norm +
        mcc_norm +
        ks_norm +
        log_loss_norm
    ) / 4

    return round(float(score_final), 6)

# FUNÇÃO PARA PROCESSAR CADA TRIO
def processar_trio(f1, f2, f3):

    try:
        temp = df[[f1, f2, f3, TARGET_COL]].dropna()

        if temp.empty:
            return None

        X = temp[[f1, f2, f3]]
        y_real = temp[TARGET_COL]

        if y_real.nunique() < 2:
            return None

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        inicio_gmm = time.perf_counter()

        gmm = GaussianMixture(
            n_components=numero_de_componentes,
            covariance_type=tipo_matriz_covariancia,
            random_state=estado_randomico,
            reg_covar=erro_numerico,
            n_init=inicializacoes_gausianas
        )

        gmm.fit(X_scaled)

        fim_gmm = time.perf_counter()

        clusters = gmm.predict(X_scaled)

        ct = pd.crosstab(clusters, y_real)

        if 1 not in ct.columns:
            return None

        cluster_fraude = ct[1].idxmax()

        probabilidades = gmm.predict_proba(
            X_scaled
        )[:, cluster_fraude]

        probabilidades = np.clip(
            probabilidades,
            1e-15,
            1 - 1e-15
        )

        y_pred = (
            probabilidades >= THRESHOLD
        ).astype(int)

        precision_vals, recall_vals, _ = precision_recall_curve(
            y_real,
            probabilidades
        )

        auc_pr = auc(recall_vals, precision_vals)

        mcc = matthews_corrcoef(
            y_real,
            y_pred
        )

        ks = ks_2samp(
            probabilidades[y_real == 0],
            probabilidades[y_real == 1]
        ).statistic

        ll = log_loss(
            y_real,
            probabilidades
        )

        score_final = calcular_score_final(
            auc_pr,
            mcc,
            ks,
            ll
        )

        return {
            "Combinacao": f"{f1} | {f2} | {f3}",
            "Feature_1": f1,
            "Feature_2": f2,
            "Feature_3": f3,
            "AUC_PR": round(float(auc_pr), 6),
            "MCC": round(float(mcc), 6),
            "KS": round(float(ks), 6),
            "Log_Loss": round(float(ll), 6),
            "Score_Final": score_final,
            "Tempo": round(float(fim_gmm - inicio_gmm), 6)
        }

    except Exception as e:
        print(f"ERRO -> {f1} + {f2} + {f3}: {e}")
        return None

# PARALELISMO
resultados = Parallel(
    n_jobs=N_JOBS,
    verbose=10
)(
    delayed(processar_trio)(f1, f2, f3)
    for f1, f2, f3 in combinacoes
)

# Remove erros/None
resultados = [
    r for r in resultados
    if r is not None
]

# DATAFRAME FINAL
df_resultados = pd.DataFrame(resultados)

df_resultados = df_resultados.sort_values(
    by="Score_Final",
    ascending=False
).reset_index(drop=True)

df_resultados["Posicao_Rank"] = df_resultados.index + 1

display(df_resultados)

df_resultados.to_csv(
    NOME_CSV,
    index=False
)

fim = time.time()

print("\nArquivo criado com sucesso:")
print(NOME_CSV)

print(
    f"Tempo total: "
    f"{fim - inicio:.4f} segundos finalizados")

Total de trios: 4060


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done   4 tasks      | elapsed:  2.1min
[Parallel(n_jobs=7)]: Done  11 tasks      | elapsed:  3.2min
[Parallel(n_jobs=7)]: Done  18 tasks      | elapsed:  3.7min
[Parallel(n_jobs=7)]: Done  27 tasks      | elapsed:  4.8min
[Parallel(n_jobs=7)]: Done  36 tasks      | elapsed:  5.8min
[Parallel(n_jobs=7)]: Done  47 tasks      | elapsed:  7.2min
[Parallel(n_jobs=7)]: Done  58 tasks      | elapsed:  8.4min
[Parallel(n_jobs=7)]: Done  71 tasks      | elapsed: 10.7min
[Parallel(n_jobs=7)]: Done  84 tasks      | elapsed: 12.3min
[Parallel(n_jobs=7)]: Done  99 tasks      | elapsed: 14.0min
[Parallel(n_jobs=7)]: Done 114 tasks      | elapsed: 15.3min
[Parallel(n_jobs=7)]: Done 131 tasks      | elapsed: 17.1min
[Parallel(n_jobs=7)]: Done 148 tasks      | elapsed: 18.9min
[Parallel(n_jobs=7)]: Done 167 tasks      | elapsed: 21.0min
[Parallel(n_jobs=7)]: Done 186 tasks      | elapsed: 23.8min
[Parallel(

,Combinacao,Feature_1,Feature_2,Feature_3,AUC_PR,MCC,KS,Log_Loss,Score_Final,Tempo,Posicao_Rank
0,V11 | V17 | V22,V11,V17,V22,0.560802,0.262529,0.850978,0.111827,0.735616,54.306143,1
1,V11 | V15 | V17,V11,V15,V17,0.571229,0.235140,0.850609,0.132313,0.730639,25.543523,2
2,V11 | V17 | V26,V11,V17,V26,0.575682,0.226267,0.844660,0.134569,0.728717,50.257560,3
3,V11 | V13 | V17,V11,V13,V17,0.570066,0.216291,0.846057,0.146115,0.724196,50.871020,4
4,V14 | V17 | V26,V14,V17,V26,0.641231,0.189782,0.862238,0.263699,0.722422,43.046070,5
...,...,...,...,...,...,...,...,...,...,...,...
4055,V4 | V22 | tempo_desde_a_primeira_transacao,V4,V22,tempo_desde_a_primeira_transacao,0.002058,0.004805,0.102221,7.083368,0.182598,9.025783,4056
4056,V2 | V24 | tempo_desde_a_primeira_transacao,V2,V24,tempo_desde_a_primeira_transacao,0.001949,0.002103,0.090291,6.344555,0.182362,11.009301,4057
4057,V2 | V26 | tempo_desde_a_primeira_transacao,V2,V26,tempo_desde_a_primeira_transacao,0.001878,0.002130,0.090601,6.365009,0.182330,11.151927,4058
4058,V2 | V13 | tempo_desde_a_primeira_transacao,V2,V13,tempo_desde_a_primeira_transacao,0.001913,0.003566,0.092928,6.638103,0.181887,11.182302,4059



Arquivo criado com sucesso:
3x3_visu_scores.csv
Tempo total: 17297.6345 segundos finalizados
